# Mila? Gemma 3 tag service (Kaggle)

Before running: choose a GPU accelerator and enable Internet in notebook settings. Accept the access terms for `google/gemma-3-4b-it` on Hugging Face. In Kaggle **Add-ons → Secrets**, add `HF_TOKEN` (a Hugging Face token with model access) and `JALDI_TAG_SERVICE_TOKEN` (a random secret you also set for the local Spring Boot backend). Keep this notebook running while using Generate with AI. The Cloudflare quick-tunnel URL changes when the notebook restarts.

In [ ]:
%pip install -q "transformers>=4.51,<5" accelerate fastapi uvicorn

In [ ]:
import hmac
import json
import os
import queue
import subprocess
import threading
import time
import urllib.request

import requests
import torch
import uvicorn
from fastapi import Depends, FastAPI, Header, HTTPException
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
from pydantic import BaseModel, Field
from transformers import AutoModelForCausalLM, AutoTokenizer

secrets_client = UserSecretsClient()
hf_token = secrets_client.get_secret("HF_TOKEN")
service_token = secrets_client.get_secret("JALDI_TAG_SERVICE_TOKEN")
if not hf_token or not service_token:
    raise RuntimeError("Set both HF_TOKEN and JALDI_TAG_SERVICE_TOKEN in Kaggle Add-ons → Secrets.")
if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU accelerator in Kaggle notebook settings, then restart the session.")

login(token=hf_token)
model_id = "google/gemma-3-4b-it"
tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    token=hf_token,
    torch_dtype=torch.float16,
    device_map="auto",
).eval()

LANGUAGES = [
    "english", "hinglish", "hindi", "marathi", "bengali", "assamese", "gujarati",
    "punjabi", "odia", "tamil", "telugu", "kannada", "malayalam", "urdu", "typos", "aiWords"
]
LANGUAGE_NAMES = {
    "english": "English", "hinglish": "Hinglish", "hindi": "Hindi", "marathi": "Marathi",
    "bengali": "Bengali", "assamese": "Assamese", "gujarati": "Gujarati", "punjabi": "Punjabi",
    "odia": "Odia", "tamil": "Tamil", "telugu": "Telugu", "kannada": "Kannada",
    "malayalam": "Malayalam", "urdu": "Urdu", "typos": "common misspellings",
    "aiWords": "related shopper search phrases"
}
app = FastAPI()

class ProductInput(BaseModel):
    name: str = Field(min_length=1, max_length=100)
    brand: str = Field(default="", max_length=60)
    category: str = Field(default="", max_length=60)

def require_token(authorization: str | None = Header(default=None)):
    expected = f"Bearer {service_token}"
    if authorization is None or not hmac.compare_digest(authorization, expected):
        raise HTTPException(status_code=401, detail="Unauthorized")

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/generate-tags", dependencies=[Depends(require_token)])
def generate_tags(product: ProductInput):
    prompt = (
        "You create accurate grocery search tags for Indian shoppers. Return ONLY one valid JSON object. "
        "Use exactly these keys with these language meanings: "
        + ", ".join(f"{key} ({LANGUAGE_NAMES[key]})" for key in LANGUAGES)
        + ". Each value must be an array of 0 to 5 short strings. "
        "Translate or transliterate only when you are confident; use an empty array instead of inventing a translation. "
        "English and Hinglish tags may include common search terms. Typos must be plausible misspellings. "
        "Do not invent product properties, variants, health claims, or unrelated products. "
        "Example shape: {\"english\":[\"milk\"],\"hinglish\":[\"doodh\"]}.\n"
        f"Product name: {product.name}\nBrand: {product.brand}\nCategory: {product.category}\n"
        "Generate useful tags for this exact product."
    )
    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt"
    ).to(model.device)
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=700, do_sample=False)
    answer = tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    start = answer.find("{")
    if start < 0:
        raise HTTPException(status_code=502, detail="Gemma did not return JSON tags")
    try:
        generated, _ = json.JSONDecoder().raw_decode(answer[start:])
    except json.JSONDecodeError as error:
        raise HTTPException(status_code=502, detail="Gemma returned invalid JSON tags") from error
    if not isinstance(generated, dict):
        raise HTTPException(status_code=502, detail="Gemma returned an invalid tag response")
    result = {}
    for language in LANGUAGES:
        values = generated.get(language, [])
        if isinstance(values, list):
            tags = list(dict.fromkeys(
                value.strip() for value in values
                if isinstance(value, str) and value.strip()
            ))[:5]
            if tags:
                result[language] = tags
    if not result:
        raise HTTPException(status_code=502, detail="Gemma returned no usable tags")
    return result

# Start the private API locally inside this Kaggle session.
server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
for _ in range(60):
    try:
        if requests.get("http://127.0.0.1:8000/health", timeout=2).ok:
            break
    except requests.RequestException:
        time.sleep(2)
else:
    raise RuntimeError("Gemma API did not start")

# A quick tunnel exposes only this notebook API; the endpoint still checks the shared bearer token.
cloudflared = "/tmp/cloudflared"
urllib.request.urlretrieve(
    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
    cloudflared,
)
os.chmod(cloudflared, 0o755)
tunnel = subprocess.Popen(
    [cloudflared, "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
lines = queue.Queue()
threading.Thread(target=lambda: [lines.put(line) for line in tunnel.stdout], daemon=True).start()
public_url = None
deadline = time.time() + 90
while time.time() < deadline and tunnel.poll() is None:
    try:
        line = lines.get(timeout=1)
    except queue.Empty:
        continue
    if "trycloudflare.com" in line:
        import re
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            public_url = match.group(0)
            break
if public_url is None:
    tunnel.terminate()
    raise RuntimeError("Could not start the Cloudflare tunnel. Check Kaggle Internet settings and notebook output.")

print("Gemma API is ready. Set GEMMA_TAG_SERVICE_URL to:", public_url)
print("Keep this notebook cell running; stop the cell to shut down the service.")
tunnel.wait()